In [3]:
"""
Phase 2 - Commit 1
Loads the raw F1 dataset, constructs the binary podium target, 
applies minimal median imputation to numeric columns only, and produces an 
isolated raw train/test split for the Task 1 unmodified-eLCS floor run. 
No leakage remedition happens here - that will be done in later stages.
"""

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from skeLCS import eLCS

RAW_DATA_PATH = "f1_raw_25299773.csv"

# Load the raw F1 dataset

df_raw = pd.read_csv(RAW_DATA_PATH, na_values="\\N", low_memory=False)

df_raw["podium"] = df_raw["positionOrder"].between(1, 3).astype(int) # Setting the target variable to 1 if the driver finished in the top 3, otherwise 0

# Minimal column selection to avoid leakage and reduce dimensionality. Exclude columns that are identifiers or directly related to the target variable.

excluded_cols = {
    "position", "positionText", "positionOrder",
    "resultId", "driverId", "raceId", "constructorId", "circuitId",
    "number", "number_driver", "number_quali",
    "podium",
}

numeric_cols = df_raw.select_dtypes(include=[np.number]).columns
feature_cols_raw = [c for c in numeric_cols if c not in excluded_cols]

X_raw = df_raw[feature_cols_raw].copy()
y_raw = df_raw["podium"].copy()

# Minimal missing value imputation for numeric columns only, using median imputation.

X_raw = X_raw.fillna(X_raw.median(numeric_only=True))

print("Raw feature matrix shape:", X_raw.shape)
print("Podium class balance:\n", y_raw.value_counts(normalize=True).round(4))

# Isolated 80/20 train/test split for the Task 1 unmodified-eLCS floor run. No leakage remediation is applied here.

X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
    X_raw, y_raw,
    test_size=0.2,
    stratify=y_raw,
    random_state=42,
)

print("X_train_raw:", X_train_raw.shape, "| X_test_raw:", X_test_raw.shape)



Raw feature matrix shape: (26759, 24)
Podium class balance:
 podium
0    0.8731
1    0.1269
Name: proportion, dtype: float64
X_train_raw: (21407, 24) | X_test_raw: (5352, 24)
